# Can we predict next-month churn?
Target: `churn_next_month` (monthly account snapshots, 3,000 accounts). Plan: audit the data (especially leakage), build honest validation, compare a simple baseline to gradient boosting, and report what the result means.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
df = pd.read_csv('data.csv')
print(df.shape, df.account_ref.nunique(), 'accounts;', df.isna().sum().sum(), 'missing values')
print(df.snapshot_month.min(), '->', df.snapshot_month.max())
print('churn rate', df.churn_next_month.mean().round(4))
df.describe().T

## Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from the CRM export of 2026-01-05*. That is measured after the snapshot and can encode the outcome. Compare it with `tenure_months`.

In [ ]:
df['gap'] = df.lifetime_months - df.tenure_months
print(df.groupby('churn_next_month').gap.describe())
print('Churners with lifetime==tenure:', (df[df.churn_next_month==1].gap==0).mean())
print('Non-churners with lifetime==tenure:', (df[df.churn_next_month==0].gap==0).mean())

For every churn row `lifetime_months == tenure_months` (gap exactly 0); for non-churners the gap is almost always positive. The column reveals when the account ended, i.e. the future. **It is excluded.** A model using it would look near-perfect and be useless at prediction time. All remaining features are known in the snapshot month.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
df.groupby('tenure_months').churn_next_month.mean().plot(ax=ax[0], title='Churn rate by tenure')
df.groupby(pd.qcut(df.usage_hours, 10, duplicates='drop'), observed=True).churn_next_month.mean().plot(ax=ax[1], title='Churn rate by usage decile', rot=45)
df.groupby('support_tickets').churn_next_month.mean().plot(ax=ax[2], title='Churn rate by tickets')
plt.tight_layout(); plt.show()
print(df.groupby('plan').churn_next_month.agg(['mean', 'size']))

## Validation design
Accounts appear in many monthly rows, so a random row split would leak the same account across train and test. I use **GroupKFold by account**, plus a **temporal holdout** (train on earlier months, test on the last 4) to check stability over time.

In [ ]:
feats = ['plan', 'seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets']
X = pd.get_dummies(df[feats], columns=['plan'], dtype=float)
y = df.churn_next_month; g = df.account_ref
models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'hist_gb': HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4, random_state=0),
}
oof = {k: np.zeros(len(df)) for k in models}
for tr, te in GroupKFold(5).split(X, y, g):
    for k, m in models.items():
        m.fit(X.iloc[tr], y.iloc[tr]); oof[k][te] = m.predict_proba(X.iloc[te])[:, 1]
rows = [{'model': 'no skill (prevalence)', 'ROC AUC': 0.5, 'PR AUC': y.mean()}]
for k, p in oof.items():
    rows.append({'model': k, 'ROC AUC': roc_auc_score(y, p), 'PR AUC': average_precision_score(y, p)})
pd.DataFrame(rows).round(3)

In [ ]:
# Temporal holdout: train before the last 4 months, test on them
cut = sorted(df.snapshot_month.unique())[-4]
tr = (df.snapshot_month < cut).values; te = ~tr
res = []
for k, m in models.items():
    m.fit(X[tr], y[tr]); p = m.predict_proba(X[te])[:, 1]
    res.append({'model': k, 'test from': cut, 'ROC AUC': roc_auc_score(y[te], p),
                'PR AUC': average_precision_score(y[te], p), 'test base rate': y[te].mean()})
pd.DataFrame(res).round(3)

In [ ]:
# Lift and calibration of out-of-fold predictions (gradient boosting)
d = pd.DataFrame({'p': oof['hist_gb'], 'y': y})
d['decile'] = pd.qcut(d.p.rank(method='first'), 10, labels=False) + 1
lift = d.groupby('decile').y.agg(['mean', 'sum']); lift['lift'] = lift['mean'] / y.mean()
print(lift.round(3))
d['bin'] = pd.qcut(d.p, 10, duplicates='drop')
d.groupby('bin', observed=True).agg(pred=('p', 'mean'), actual=('y', 'mean')).round(3)

In [ ]:
from sklearn.inspection import permutation_importance
tr_i, te_i = next(GroupKFold(5).split(X, y, g))
m = models['hist_gb'].fit(X.iloc[tr_i], y.iloc[tr_i])
r = permutation_importance(m, X.iloc[te_i], y.iloc[te_i], scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False).round(4)

## Conclusions
- **Leakage:** `lifetime_months` equals `tenure_months` for 100% of churn rows and 0% of non-churn rows, so it was excluded.
- **Signal is real but modest.** Account-grouped 5-fold CV: ROC AUC 0.66 (logistic) and 0.68 (gradient boosting) against 0.50 for no skill. PR AUC is 0.295 against a 0.154 base rate. On a temporal holdout (last 4 months) ROC AUC is about 0.71 for both models.
- **Usable for ranking, not for confident calls.** The top-scored decile churns at about 36% (2.3x lift), and the bottom decile at about 6.5%. Probabilities are well calibrated. No threshold would give high precision and recall together.
- **Drivers** (permutation importance): `support_tickets`, `usage_hours`, then `tenure_months`. `plan` adds nothing beyond fee and seats.
- **Caveats:** the churn base rate falls from 15% overall to 9% in the holdout months, so check for right-censoring or a real shift before trusting absolute rates. The features are a single snapshot; month-over-month trends in usage and tickets are the obvious next features. Test AUC differences between the two models are within noise.
- **Environment note:** cell outputs were generated by running the cells in a plain Python process, not a Jupyter kernel (the sandbox blocks the kernel's local ports). Re-running in Jupyter should give the same numbers.